In [1]:
import numpy as np
import pandas as pd
import os
import glob

In [2]:
DATA_DIR = "data"
WINDOW_SIZE = 24
TARGET_FLAT_PCT = 0.60 

alphas_range = np.arange(0.1, 2.05, 0.05)

In [3]:
search_path = os.path.join(DATA_DIR, "*_5min.parquet")
files = glob.glob(search_path)
tickers_to_test = [os.path.basename(f).replace("_5min.parquet", "") for f in files]

print(f"Найдено тикеров для анализа: {len(tickers_to_test)}")

Найдено тикеров для анализа: 16


In [4]:
def fill_time_gaps(df: pd.DataFrame, interval_name: str = "5min") -> pd.DataFrame:
    resample_map = {"5min": "5T", "15min": "15T", "1hour": "H", "1day": "D"}
    freq = resample_map.get(interval_name, "5T")
    
    df = df.copy()
    
    if 'DateTime' in df.columns:
        df['DateTime'] = pd.to_datetime(df['DateTime'])
        df = df.set_index('DateTime')
    elif not isinstance(df.index, pd.DatetimeIndex):
        raise ValueError("DataFrame должен иметь колонку 'DateTime' или DatetimeIndex")
    
    df = df.sort_index()
    
    full_range = pd.date_range(start=df.index.min(), end=df.index.max(), freq=freq)
    df = df.reindex(full_range)
    df.index.name = "DateTime"
    
    price_cols = ['Open', 'High', 'Low', 'Close']
    df[price_cols] = df[price_cols].ffill().bfill()
    if 'Volume' in df.columns:
        df['Volume'] = df['Volume'].fillna(0)
    
    # Гарантированно возвращаем DateTime как колонку, а не как индекс
    return df.reset_index()

In [5]:
def clean_market_data(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    
    if 'DateTime' in df.columns:
        df = df.set_index('DateTime').sort_index()
    elif isinstance(df.index, pd.DatetimeIndex):
        df = df.sort_index()
        if df.index.name is None:
            df.index.name = 'DateTime'
    else:
        raise ValueError("Нет колонки или индекса DateTime")
    
    cols_to_fix = ['Open', 'High', 'Low', 'Close']
    
    for col in cols_to_fix:
        if col in df.columns:
            df[col] = df[col].replace(0, pd.NA)
    
    df[cols_to_fix] = df[cols_to_fix].ffill()
    df[cols_to_fix] = df[cols_to_fix].bfill()
    
    if 'Volume' in df.columns:
        df['Volume'] = df['Volume'].fillna(0)
    
    # Гарантированно возвращаем DateTime как колонку
    return df.reset_index()

In [6]:
results_list = []

for i, ticker in enumerate(tickers_to_test):
    file_path = os.path.join(DATA_DIR, f"{ticker}_5min.parquet")
    
    try:
        df = pd.read_parquet(file_path)
        df = df[df["DateTime"] >= "2020-01-01"]
        df = fill_time_gaps(df)
        df = clean_market_data(df)
        
        close = df['Close']
        
        future_return = (close.shift(-WINDOW_SIZE) - close) / close
        returns_5m = close.pct_change()
        volatility_5m = returns_5m.rolling(window=WINDOW_SIZE).std()
        volatility_2h_equiv = volatility_5m * np.sqrt(WINDOW_SIZE)
        
        valid_idx = future_return.notna() & volatility_2h_equiv.notna() & (volatility_2h_equiv > 1e-7)
        future_return_valid = future_return[valid_idx]
        volatility_valid = volatility_2h_equiv[valid_idx]
        
        if len(future_return_valid) < 10000: # Пропускаем тикеры с слишком маленькой историей
            continue
            
        best_alpha = None
        min_diff = 999
        total_valid = len(future_return_valid)
        
        for alpha in alphas_range:
            threshold_up = alpha * volatility_valid
            threshold_down = -alpha * volatility_valid
            
            is_up = future_return_valid >= threshold_up
            is_down = future_return_valid <= threshold_down
            is_flat = ~is_up & ~is_down
            
            flat_pct = is_flat.sum() / total_valid
            
            diff = abs(flat_pct - TARGET_FLAT_PCT)
            if diff < min_diff:
                min_diff = diff
                best_alpha = alpha
                best_flat_pct = flat_pct
                best_up_pct = is_up.sum() / total_valid
                best_down_pct = is_down.sum() / total_valid
                
        results_list.append({
            'Ticker': ticker,
            'Best_Alpha': best_alpha,
            'Flat%': best_flat_pct * 100,
            'Up%': best_up_pct * 100,
            'Down%': best_down_pct * 100
        })
        
        # Выводим прогресс каждые 20 тикеров
        if (i + 1) % 20 == 0 or (i + 1) == len(tickers_to_test):
            print(f"Обработано {i + 1}/{len(tickers_to_test)} тикеров...")
        
    except Exception as e:
        print(f"❌ Ошибка {ticker}: {e}")

/var/folders/dk/wsx6v8916pz0_nfsc908f2_h0000gn/T/ipykernel_24037/3624895608.py:15: FutureWarning: 'T' is deprecated and will be removed in a future version, please use 'min' instead.
  full_range = pd.date_range(start=df.index.min(), end=df.index.max(), freq=freq)
/var/folders/dk/wsx6v8916pz0_nfsc908f2_h0000gn/T/ipykernel_24037/3624895608.py:15: FutureWarning: 'T' is deprecated and will be removed in a future version, please use 'min' instead.
  full_range = pd.date_range(start=df.index.min(), end=df.index.max(), freq=freq)
/var/folders/dk/wsx6v8916pz0_nfsc908f2_h0000gn/T/ipykernel_24037/3624895608.py:15: FutureWarning: 'T' is deprecated and will be removed in a future version, please use 'min' instead.
  full_range = pd.date_range(start=df.index.min(), end=df.index.max(), freq=freq)
/var/folders/dk/wsx6v8916pz0_nfsc908f2_h0000gn/T/ipykernel_24037/3624895608.py:15: FutureWarning: 'T' is deprecated and will be removed in a future version, please use 'min' instead.
  full_range = pd.date

Обработано 16/16 тикеров...


/var/folders/dk/wsx6v8916pz0_nfsc908f2_h0000gn/T/ipykernel_24037/3624895608.py:15: FutureWarning: 'T' is deprecated and will be removed in a future version, please use 'min' instead.
  full_range = pd.date_range(start=df.index.min(), end=df.index.max(), freq=freq)


In [7]:
# === ФИНАЛЬНЫЙ АНАЛИЗ ===
if results_list:
    res_df = pd.DataFrame(results_list)
    
    mean_alpha = res_df['Best_Alpha'].mean()
    median_alpha = res_df['Best_Alpha'].median()
    mode_alpha = res_df['Best_Alpha'].mode().iloc[0] if not res_df['Best_Alpha'].mode().empty else None
    
    print("\n" + "="*70)
    print(f"ИТОГОВЫЙ АНАЛИЗ ОПТИМАЛЬНОЙ ALPHA ДЛЯ РЫНКА РФ (Всего тикеров: {len(res_df)})")
    print("="*70)
    print(f"Средняя оптимальная Alpha:   {mean_alpha:.4f}")
    print(f"Медианная оптимальная Alpha: {median_alpha:.4f}")
    print(f"Модальная оптимальная Alpha: {mode_alpha}")
    print("-"*70)
    print("Распределение найденных Alpha:")
    print(res_df['Best_Alpha'].describe())
    print("="*70)
    
    # Выводим топ-10 и анти-топ-10 для интереса
    print("\nТикеры с НАИМЕНЬШЕЙ альфой (самые шумные/волатильные):")
    print(res_df.nsmallest(5, 'Best_Alpha')[['Ticker', 'Best_Alpha', 'Flat%']].to_string(index=False))
    
    print("\nТикеры с НАИБОЛЬШЕЙ альфой (самые трендовые/спокойные):")
    print(res_df.nlargest(5, 'Best_Alpha')[['Ticker', 'Best_Alpha', 'Flat%']].to_string(index=False))
else:
    print("Нет данных для анализа.")


ИТОГОВЫЙ АНАЛИЗ ОПТИМАЛЬНОЙ ALPHA ДЛЯ РЫНКА РФ (Всего тикеров: 16)
Средняя оптимальная Alpha:   0.4625
Медианная оптимальная Alpha: 0.4500
Модальная оптимальная Alpha: 0.45000000000000007
----------------------------------------------------------------------
Распределение найденных Alpha:
count    16.000000
mean      0.462500
std       0.028868
min       0.450000
25%       0.450000
50%       0.450000
75%       0.450000
max       0.550000
Name: Best_Alpha, dtype: float64

Тикеры с НАИМЕНЬШЕЙ альфой (самые шумные/волатильные):
Ticker  Best_Alpha     Flat%
  YDEX        0.45 59.781679
  TATN        0.45 59.777293
  GAZP        0.45 61.640323
  RTKM        0.45 61.206166
  ALRS        0.45 60.559756

Тикеры с НАИБОЛЬШЕЙ альфой (самые трендовые/спокойные):
    Ticker  Best_Alpha     Flat%
CNYRUB_TOM        0.55 60.201131
GLDRUB_TOM        0.50 59.788464
      BRH6        0.50 59.401549
      YDEX        0.45 59.781679
      TATN        0.45 59.777293
